# Chess-Coach Agent — Gemma 4 E4B QLoRA on Kaggle (2×T4)

Aimed single-session run. LoRA adapter on the v1.2 agentic-harness corpus (skills + tools +
the `python` verify tool; fast/think/auto). Download/pull the adapter → serve on the RTX 4060.

## Just run it
1. Settings → Accelerator → **GPU T4 ×2**.  2. Internet **On**.
3. Add-ons → Secrets → **HF_TOKEN** (WRITE scope — base download + checkpoint push).
4. Accept the Gemma 4 license once.
5. **Run top → bottom.** No edits needed — Cell 1 is pre-filled. If Kaggle shows a RESTART
   banner after Cell 4 (deps), restart, then continue.

## What this run is
`all-linear` targets + **FORMAT_WEIGHT=8.0** on the control tags (already in the loss) — the
fix for the free-gen format collapse, proven by the micro-overfit gate (8/8 emit our tags).
LR 1e-4 + ~600 updates (the prior run overfit at ~step 400, so gentler + shorter). ~9–11h =
ONE session. `best/` (lowest val) auto-mirrors to CKPT_REPO every eval, so a 12h kill can't
lose it; if a session ends early, just re-run — Cell 4.5 resumes automatically.

## Watch (the only two signals)
- The model should emit clean `<skill>`/`<tool>` within the first ~100 steps (format fixed).
- `val_loss` (every 100) should drop; `best/` saves the lowest. Serve `best/`.

## OOM ladder (never cut seq — it's the data floor)
RANK 16→8 (keep all-linear) → DDP=False (1 GPU). seq stays 1664.


In [ ]:
# Cell 1 — config (E4B QLoRA, Kaggle 2×T4 DDP). FINAL: pre-filled, run top-to-bottom, no edits.
import os

MODEL  = "gemma4_e4b"
ENGINE = "unsloth"
HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False   # E4B MUST be 4-bit

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_kaggle"
DATA_STEM = "v1_2"   # FULL 73k corpus

MAX_SEQ = 1664          # data floor (corpus max 1642); below this truncates finals
TARGETS = "all-linear"  # attn + MLP — REQUIRED so the model can EMIT the <skill>/<tool> format
RANK = 16
GRAD_ACCUM = 16
DDP = True              # 2×T4: one full replica per GPU, grads all-reduced, ~2x throughput

# AIMED single-session run. The format fix (FORMAT_WEIGHT=8.0 on the control tags) is already
# in the loss and was proven by the micro-overfit gate (8/8 emit our tags). LR is halved and
# the horizon short because the prior run overfit at ~step 400 — run gentler, stop early.
# ~500-600 updates ≈ ~9-11h on 2×T4 = ONE Kaggle session.
LR = 1e-4
MAX_STEPS = 600
EVAL_EVERY = 100        # val + best/ save cadence; format should emit clean by ~step 100
MAX_VAL = 128
SAVE_EVERY = 50

# Off-kernel mirror: best/ (lowest val → SERVE) + checkpoint/ pushed here every save, and
# auto-pulled to resume / to serve (no zip). FRESH v3 repo — the loss changed, so do NOT
# reuse the v2 (old-loss) checkpoints. (Clear to "" for local-only, but then a 12h kill
# before MAX_STEPS loses progress.)
CKPT_REPO = "RyanDev1st/gemma4-chesscoach-ckpt-v3"
RESUME = False          # auto-managed by Cell 4.5; leave False
RESUME_DATASET = ""     # (legacy) unused when CKPT_REPO is set
if CKPT_REPO:
    os.environ["CHESS_CKPT_REPO"] = CKPT_REPO   # picked up by train_unsloth _hub_push

print("config:", MODEL, ENGINE, "| seq", MAX_SEQ, "| targets", TARGETS, "| rank", RANK,
      "| lr", LR, "| steps", MAX_STEPS, "| ckpt", CKPT_REPO or "(local only)")


In [ ]:
# Cell 2 — GPU preflight. If this fails: Settings (right panel) → Accelerator → GPU T4 ×2,
# then re-run. A CPU kernel has no `nvidia-smi` and cannot train E4B.
import subprocess, shutil, torch
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                          "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi NOT found — no GPU attached to this kernel.")
assert torch.cuda.is_available(), (
    "No GPU — Settings → Accelerator → GPU T4 ×2 (×2 needed for DDP), then re-run. "
    "Batch/commit runs default to CPU; you MUST select the accelerator before running.")
n = torch.cuda.device_count()
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0), "| count", n)
if n < 2:
    print("\nℹ️  Only 1 GPU — set DDP=False in Cell 1, or pick GPU T4 ×2 for the ~2x DDP speedup.")


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps. Let Unsloth resolve its own current stack (do NOT pin an old
# transformers — Gemma 4 needs a recent one).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    print("unsloth installed. If Kaggle shows a RESTART banner, restart, then run Cells 5+. "
          "If Gemma 4 fails to load with a model-type error, run: pip install -U transformers")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 4.5 — HF login + AUTO-RESUME pull + CONFIRM (runs BEFORE the base-model download
# and fit tests). Catches a bad/empty checkpoint in ~30s instead of after minutes of
# weight loading. Pulls checkpoint/ (latest, resume) + best/ (lowest-val, serve) from
# CKPT_REPO and SETS `RESUME` from what actually lands on disk. Re-run the notebook each
# session; this continues automatically — no flag, no download, no Kaggle Dataset.
import os, glob, shutil
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))   # also authenticates Cell 5 base download

dst = f"{WORKDIR}/runs/{OUTPUT}"
os.makedirs(dst, exist_ok=True)

def _has_state(d):
    return os.path.exists(f"{d}/checkpoint/trainer_state.pt")

if CKPT_REPO:
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=CKPT_REPO, repo_type="model",
                          allow_patterns=["checkpoint/*", "best/*"], local_dir=dst)
        print("Hub pull OK:", CKPT_REPO)
    except Exception as e:
        print("Hub pull skipped (", repr(e), ") — fresh start unless a local checkpoint exists")
elif RESUME:
    # Legacy fallback: stage from a Kaggle Dataset (zip or folder) in RESUME_DATASET.
    src = RESUME_DATASET
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        print("unzip", zips[0], "->", dst); shutil.unpack_archive(zips[0], dst)
    elif os.path.isdir(src):
        for sub in ("checkpoint", "best"):
            s = os.path.join(src, sub)
            if not os.path.isdir(s):
                s = next((q for q in glob.glob(f"{src}/**/{sub}", recursive=True)), None)
            if s and os.path.isdir(s):
                shutil.copytree(s, os.path.join(dst, sub), dirs_exist_ok=True)
                print("copied", s, "->", os.path.join(dst, sub))

# CONFIRM before any weight loading: resume only if BOTH the trainer state AND the adapter
# weights are present (exactly what train_unsloth._load_ckpt needs). Report the step.
RESUME = False
ck = f"{dst}/checkpoint"
adapter = os.path.exists(f"{ck}/adapter_model.safetensors") or os.path.exists(f"{ck}/adapter_model.bin")
if _has_state(dst) and adapter:
    import torch
    st = torch.load(f"{ck}/trainer_state.pt", map_location="cpu")
    bv = st.get("best_val"); bv = f"{bv:.4f}" if isinstance(bv, (int, float)) else bv
    RESUME = True
    print(f"✅ CONFIRMED resume @ update {st.get('update')}/{MAX_STEPS} "
          f"(epoch {st.get('epoch')}, best_val {bv}). Safe to load weights / train.")
    print("   checkpoint files:", os.listdir(ck))
    if os.path.isdir(f"{dst}/best"):
        print("   best/ present (serve):", os.listdir(f"{dst}/best"))
elif _has_state(dst) and not adapter:
    raise SystemExit("checkpoint/trainer_state.pt present but adapter weights MISSING — bad "
                     "pull. Fix CKPT_REPO before wasting time on the base-model download.")
else:
    print("No checkpoint — FRESH run (first session). RESUME=False.")


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in (f"{DATA_STEM}_train.jsonl", f"{DATA_STEM}_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 7 - TRAIN (E4B QLoRA, Unsloth, all-linear, format-weighted loss). The ONLY cell you
# wait on. DDP=True -> accelerate launch 2 procs (~2x). Checkpoints every SAVE_EVERY + on
# timeout/crash; best/ = lowest val (-> SERVE). With CKPT_REPO set, best/ + checkpoint/ mirror
# to the Hub each save, so a 12h kill can't lose progress and a re-run resumes automatically.
import subprocess, sys, os
args = ['--model', MODEL, '--engine', ENGINE, '--max-steps', str(MAX_STEPS),
        '--rank', str(RANK), '--targets', TARGETS, '--grad-accum', str(GRAD_ACCUM),
        '--lr', str(LR), '--max-seq', str(MAX_SEQ), '--eval-every', str(EVAL_EVERY),
        '--max-val', str(MAX_VAL), '--save-every', str(SAVE_EVERY),
        '--data-stem', DATA_STEM, '--output', OUTPUT]
if NO_4BIT: args.append('--no-4bit')
if RESUME:  args.append('--resume')
if DDP:
    cmd = ['accelerate','launch','--num_processes','2','--multi_gpu','-m','llm_training.run_train'] + args
else:
    cmd = [sys.executable,'-m','llm_training.run_train'] + args
print('>', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm',
                    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})


In [ ]:
# Cell 8 — package adapter for download (OPTIONAL; run after training OR a timeout).
# Resume is now AUTOMATIC via the Hub (Cell 4.5 pulls+confirms before weight loading), so
# you do NOT need this zip to continue — it's just a convenient local copy of the
# final/best adapter to pull onto the 4060. runs/OUTPUT has: best/ (lowest-val -> SERVE) +
# checkpoint/ (latest, for resume) + the final adapter. With CKPT_REPO set, best/ and
# checkpoint/ are already on the Hub.
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("run dir contents:", os.listdir(run_dir) if os.path.isdir(run_dir) else "MISSING")
out_zip = f"/kaggle/working/{OUTPUT}"
shutil.make_archive(out_zip, "zip", run_dir)
sz = os.path.getsize(out_zip + ".zip") / 1e6
print(f"\n✅ {out_zip}.zip ({sz:.1f} MB) — download from the Output panel (optional).")
print("   SERVE  -> the best/ folder inside (lowest val), or pull best/ from CKPT_REPO.")
print("   RESUME -> AUTOMATIC next session: just re-run the notebook (Cell 4.5 pulls the")
print("             latest checkpoint from CKPT_REPO and confirms it). No upload needed.")
